# 02_preprocessing_strategy_tests_v5_3cat_compare

Objectif : préparer une vraie V5 ciblée sur trois catégories.

Cette version répond à deux questions en même temps.

1. Sur STEEL, est ce que le tiling casse le signal ?
2. Sur Casting_class1 et cable, quelle stratégie donne le meilleur compromis entre conservation du défaut et coût de calcul ?

Le notebook génère une matrice de runs pour PatchCore V5 avec trois stratégies par catégorie.

## 1. Imports

In [2]:
##############
#  Imports   #
##############

import os
import re
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt

try:
    from scipy import ndimage
    SCIPY_AVAILABLE = True
except Exception as error:
    SCIPY_AVAILABLE = False
    ndimage = None
    print("scipy indisponible:", error)

warnings.filterwarnings("ignore")


## 2. Configuration

In [3]:
##########################
#  Global configuration  #
##########################

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

RUN_TAG = "preprocessing_v5_3cat_compare"
RUN_RESULTS_DIR = RESULTS_DIR / RUN_TAG
RUN_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

UNIFIED_CSV = PROCESSED_DIR / "unified_dataset.csv"

MASK_THRESHOLD = 127
MIN_COMPONENT_PX = 5
MIN_VISIBLE_DEFECT_PX = 12
LOW_SURVIVAL_LIMIT = 0.35

TARGET_PAIRS = [
    ("hss-iad", "STEEL"),
    ("hss-iad", "Casting_class1"),
    ("mvtec", "cable"),
]

STRATEGY_MATRIX = [
    {
        "run_id": "steel_resize_512",
        "dataset": "hss-iad",
        "category": "STEEL",
        "patchcore_mode": "full_image",
        "strategy": "resize_512",
        "full_image_strategy": "resize_512",
        "tile_strategy": "",
        "tile_size": np.nan,
        "tile_overlap": np.nan,
        "tile_resize": np.nan,
        "hypothesis": "STEEL full resize, check if tiling was hurting the signal",
    },
    {
        "run_id": "steel_letterbox_edge_512",
        "dataset": "hss-iad",
        "category": "STEEL",
        "patchcore_mode": "full_image",
        "strategy": "letterbox_edge_512",
        "full_image_strategy": "letterbox_edge_512",
        "tile_strategy": "",
        "tile_size": np.nan,
        "tile_overlap": np.nan,
        "tile_resize": np.nan,
        "hypothesis": "STEEL full letterbox edge, preserve geometry without black borders",
    },
    {
        "run_id": "steel_tile_512",
        "dataset": "hss-iad",
        "category": "STEEL",
        "patchcore_mode": "tiling",
        "strategy": "tile_512_overlap_0_50_resize_384",
        "full_image_strategy": "resize_512",
        "tile_strategy": "tile_512_overlap_0_50_resize_384",
        "tile_size": 512,
        "tile_overlap": 0.50,
        "tile_resize": 384,
        "hypothesis": "STEEL tiling baseline from V4 fast",
    },
    {
        "run_id": "casting1_resize_512",
        "dataset": "hss-iad",
        "category": "Casting_class1",
        "patchcore_mode": "full_image",
        "strategy": "resize_512",
        "full_image_strategy": "resize_512",
        "tile_strategy": "",
        "tile_size": np.nan,
        "tile_overlap": np.nan,
        "tile_resize": np.nan,
        "hypothesis": "Casting class 1 full resize, check normal versus anomaly separation",
    },
    {
        "run_id": "casting1_letterbox_edge_512",
        "dataset": "hss-iad",
        "category": "Casting_class1",
        "patchcore_mode": "full_image",
        "strategy": "letterbox_edge_512",
        "full_image_strategy": "letterbox_edge_512",
        "tile_strategy": "",
        "tile_size": np.nan,
        "tile_overlap": np.nan,
        "tile_resize": np.nan,
        "hypothesis": "Casting class 1 letterbox edge, preserve geometry",
    },
    {
        "run_id": "casting1_tile_256",
        "dataset": "hss-iad",
        "category": "Casting_class1",
        "patchcore_mode": "tiling",
        "strategy": "tile_256_overlap_0_50_resize_384",
        "full_image_strategy": "resize_512",
        "tile_strategy": "tile_256_overlap_0_50_resize_384",
        "tile_size": 256,
        "tile_overlap": 0.50,
        "tile_resize": 384,
        "hypothesis": "Casting class 1 small tiles, preserve small local defects",
    },
    {
        "run_id": "cable_resize_512",
        "dataset": "mvtec",
        "category": "cable",
        "patchcore_mode": "full_image",
        "strategy": "resize_512",
        "full_image_strategy": "resize_512",
        "tile_strategy": "",
        "tile_size": np.nan,
        "tile_overlap": np.nan,
        "tile_resize": np.nan,
        "hypothesis": "Cable full resize, baseline without tiling",
    },
    {
        "run_id": "cable_letterbox_edge_512",
        "dataset": "mvtec",
        "category": "cable",
        "patchcore_mode": "full_image",
        "strategy": "letterbox_edge_512",
        "full_image_strategy": "letterbox_edge_512",
        "tile_strategy": "",
        "tile_size": np.nan,
        "tile_overlap": np.nan,
        "tile_resize": np.nan,
        "hypothesis": "Cable letterbox edge, reduce geometry distortion",
    },
    {
        "run_id": "cable_tile_512",
        "dataset": "mvtec",
        "category": "cable",
        "patchcore_mode": "tiling",
        "strategy": "tile_512_overlap_0_50_resize_384",
        "full_image_strategy": "resize_512",
        "tile_strategy": "tile_512_overlap_0_50_resize_384",
        "tile_size": 512,
        "tile_overlap": 0.50,
        "tile_resize": 384,
        "hypothesis": "Cable tiling, check if local patches reduce heat on normal structure",
    },
]

OUTPUT_RULES_CSV = RESULTS_DIR / "preprocessing_rules_v5_3cat_compare.csv"
OUTPUT_RULES_PATCHCORE_CSV = RESULTS_DIR / "preprocessing_rules_v5_for_patchcore.csv"
OUTPUT_MASK_DIAG_CSV = RESULTS_DIR / "preprocessing_mask_diagnostics_v5_3cat_compare.csv"
OUTPUT_SUMMARY_CSV = RESULTS_DIR / "preprocessing_strategy_summary_v5_3cat_compare.csv"

print("Project root:", PROJECT_ROOT)
print("Unified CSV:", UNIFIED_CSV)
print("Run dir:", RUN_RESULTS_DIR)
print("Target pairs:", TARGET_PAIRS)


Project root: /home/nat/Documents/AI-Lab/mar26_bds_anomalies_pieces_indus
Unified CSV: /home/nat/Documents/AI-Lab/mar26_bds_anomalies_pieces_indus/data/processed/unified_dataset.csv
Run dir: /home/nat/Documents/AI-Lab/mar26_bds_anomalies_pieces_indus/notebooks/results/preprocessing_v5_3cat_compare
Target pairs: [('hss-iad', 'STEEL'), ('hss-iad', 'Casting_class1'), ('mvtec', 'cable')]


## 3. Chargement du dataset

In [4]:
####################
#  Load CSV files  #
####################

if not UNIFIED_CSV.exists():
    raise FileNotFoundError(f"Missing file: {UNIFIED_CSV}")

df = pd.read_csv(UNIFIED_CSV)

required_cols = {"dataset", "category", "split", "image_path", "mask_path"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing columns in unified dataset: {sorted(missing_cols)}")

if "is_anomaly" not in df.columns:
    if "label" in df.columns:
        label_text = df["label"].astype(str).str.lower()
        df["is_anomaly"] = ~label_text.isin(["good", "normal", "0", "false"])
    else:
        raise ValueError("Missing both is_anomaly and label columns")

df["is_anomaly"] = df["is_anomaly"].astype(bool)
df["has_mask"] = df["mask_path"].notna() & (df["mask_path"].astype(str).str.len() > 0)

available_pairs = sorted(set(zip(df["dataset"], df["category"])))
resolved_pairs = []

for wanted_dataset, wanted_category in TARGET_PAIRS:
    wanted = (wanted_dataset, wanted_category)

    if wanted in available_pairs:
        resolved_pairs.append(wanted)
        continue

    candidates = []
    for dataset, category in available_pairs:
        if category == wanted_category:
            candidates.append((dataset, category))

    if len(candidates) == 1:
        resolved_pairs.append(candidates[0])
        print("Adjusted target pair:", wanted, "->", candidates[0])
    else:
        print("Target pair not found:", wanted)

resolved_pairs = sorted(set(resolved_pairs))
if len(resolved_pairs) != len(TARGET_PAIRS):
    print("Available pairs:")
    print(available_pairs)
    raise ValueError("Not all target pairs were resolved")

keep_rows = []
for _, row in df.iterrows():
    keep_rows.append((row["dataset"], row["category"]) in resolved_pairs)

df_target = df[keep_rows].copy().reset_index(drop=True)
mask_df = df_target[df_target["is_anomaly"] & df_target["has_mask"]].copy().reset_index(drop=True)

print("Dataset images total:", len(df))
print("Target images:", len(df_target))
print("Target masks:", len(mask_df))
print()
print("Target summary:")
display(df_target.groupby(["dataset", "category", "is_anomaly"]).size().reset_index(name="images"))


Dataset images total: 17429
Target images: 5722
Target masks: 845

Target summary:


,dataset,category,is_anomaly,images
0,hss-iad,Casting_class1,False,452
1,hss-iad,Casting_class1,True,19
2,hss-iad,STEEL,False,4143
3,hss-iad,STEEL,True,734
4,mvtec,cable,False,282
5,mvtec,cable,True,92


## 4. Fonctions de diagnostic masque

In [5]:
############################
#  Paths and image helpers #
############################


def resolve_path(path_value):
    path = Path(str(path_value))

    if path.exists():
        return path

    candidate = PROJECT_ROOT / path
    if candidate.exists():
        return candidate

    candidate = DATA_DIR / path
    if candidate.exists():
        return candidate

    return PROJECT_ROOT / path


def open_mask(path_value):
    path = resolve_path(path_value)
    return Image.open(path).convert("L")


def mask_to_bool(mask_img):
    arr = np.array(mask_img.convert("L"))
    return arr > MASK_THRESHOLD


def connected_components(mask_bool):
    if not SCIPY_AVAILABLE:
        area = int(mask_bool.sum())
        if area == 0:
            return []
        return [{"area": area, "bbox": (0, 0, mask_bool.shape[1], mask_bool.shape[0])}]

    structure = np.ones((3, 3), dtype=int)
    labels, n_labels = ndimage.label(mask_bool, structure=structure)

    components = []
    for label_id in range(1, n_labels + 1):
        ys, xs = np.where(labels == label_id)
        area = int(len(xs))

        if area < MIN_COMPONENT_PX:
            continue

        components.append({
            "area": area,
            "bbox": (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1),
        })

    return components


def component_summary(mask_img):
    mask_bool = mask_to_bool(mask_img)
    components = connected_components(mask_bool)
    areas = [comp["area"] for comp in components]

    if len(areas) == 0:
        return {
            "mask_area": int(mask_bool.sum()),
            "n_components": 0,
            "min_component_px": 0,
            "max_component_px": 0,
            "mask_h": int(mask_bool.shape[0]),
            "mask_w": int(mask_bool.shape[1]),
            "mask_ratio": 0.0,
        }

    return {
        "mask_area": int(mask_bool.sum()),
        "n_components": int(len(areas)),
        "min_component_px": int(min(areas)),
        "max_component_px": int(max(areas)),
        "mask_h": int(mask_bool.shape[0]),
        "mask_w": int(mask_bool.shape[1]),
        "mask_ratio": float(mask_bool.sum() / mask_bool.size),
    }


In [6]:
###########################
#  Strategy helpers       #
###########################


def letterbox_pil(img, size, interpolation, fill_mode="edge", is_mask=False):
    w, h = img.size
    scale = min(size / w, size / h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = img.resize((new_w, new_h), interpolation)

    if is_mask or img.mode == "L":
        canvas = Image.new("L", (size, size), 0)
    else:
        canvas = Image.new("RGB", (size, size), (0, 0, 0))

    left = (size - new_w) // 2
    top = (size - new_h) // 2
    canvas.paste(resized, (left, top))
    return canvas


def apply_full_strategy_to_mask(mask_img, strategy):
    interpolation = Image.NEAREST
    strategy = str(strategy)

    if strategy == "resize_512":
        return mask_img.resize((512, 512), interpolation)

    if strategy == "letterbox_edge_512":
        return letterbox_pil(mask_img, 512, interpolation, fill_mode="edge", is_mask=True)

    raise ValueError(f"Unknown full strategy: {strategy}")


def parse_tile_strategy(strategy):
    match = re.match(r"tile_(\d+)_overlap_(\d+)_(\d+)_resize_(\d+)", str(strategy))
    if match is None:
        raise ValueError(f"Invalid tile strategy: {strategy}")

    tile_size = int(match.group(1))
    overlap = float(match.group(2) + "." + match.group(3))
    tile_resize = int(match.group(4))
    return tile_size, overlap, tile_resize


def make_tile_coords(width, height, tile_size, overlap):
    stride = max(1, int(round(tile_size * (1.0 - overlap))))

    if width <= tile_size:
        xs = [0]
    else:
        xs = list(range(0, width - tile_size + 1, stride))
        last_x = width - tile_size
        if xs[-1] != last_x:
            xs.append(last_x)

    if height <= tile_size:
        ys = [0]
    else:
        ys = list(range(0, height - tile_size + 1, stride))
        last_y = height - tile_size
        if ys[-1] != last_y:
            ys.append(last_y)

    coords = []
    for y0 in ys:
        for x0 in xs:
            x1 = min(x0 + tile_size, width)
            y1 = min(y0 + tile_size, height)
            coords.append((x0, y0, x1, y1))

    return coords


def tile_component_coverage(mask_bool, component, coords):
    x0, y0, x1, y1 = component["bbox"]
    comp_area = component["area"]
    best_coverage = 0.0

    for tx0, ty0, tx1, ty1 in coords:
        ix0 = max(x0, tx0)
        iy0 = max(y0, ty0)
        ix1 = min(x1, tx1)
        iy1 = min(y1, ty1)

        if ix1 <= ix0 or iy1 <= iy0:
            continue

        inside = mask_bool[iy0:iy1, ix0:ix1].sum()
        coverage = inside / max(comp_area, 1)
        best_coverage = max(best_coverage, float(coverage))

    return best_coverage


## 5. Diagnostic des stratégies de la matrice

In [7]:
##################################
#  Evaluate strategy matrix      #
##################################

rules_df = pd.DataFrame(STRATEGY_MATRIX)

diag_rows = []

for _, rule in rules_df.iterrows():
    dataset = rule["dataset"]
    category = rule["category"]
    strategy = rule["strategy"]
    mode = rule["patchcore_mode"]

    part = mask_df[(mask_df["dataset"] == dataset) & (mask_df["category"] == category)].copy()

    for _, row in part.iterrows():
        mask = open_mask(row["mask_path"])
        original = component_summary(mask)

        if mode == "full_image":
            transformed_mask = apply_full_strategy_to_mask(mask, rule["full_image_strategy"])
            transformed = component_summary(transformed_mask)

            expected_scale = (transformed["mask_h"] * transformed["mask_w"]) / max(original["mask_h"] * original["mask_w"], 1)
            expected_area = original["mask_area"] * expected_scale
            area_survival = transformed["mask_area"] / max(expected_area, 1.0)

            failed = bool(
                transformed["mask_area"] == 0
                or transformed["min_component_px"] < MIN_VISIBLE_DEFECT_PX
                or area_survival < LOW_SURVIVAL_LIMIT
            )

            diag_rows.append({
                "run_id": rule["run_id"],
                "dataset": dataset,
                "category": category,
                "defect_type": row.get("defect_type", "unknown"),
                "strategy": strategy,
                "patchcore_mode": mode,
                "image_path": row["image_path"],
                "mask_path": row["mask_path"],
                "orig_mask_ratio": original["mask_ratio"],
                "orig_min_component_px": original["min_component_px"],
                "after_min_component_px": transformed["min_component_px"],
                "area_survival": float(area_survival),
                "tile_count": np.nan,
                "effective_min_component_px": np.nan,
                "failed": failed,
            })

        else:
            tile_size = int(rule["tile_size"])
            overlap = float(rule["tile_overlap"])
            tile_resize = int(rule["tile_resize"])
            mask_bool = mask_to_bool(mask)
            components = connected_components(mask_bool)
            coords = make_tile_coords(mask.size[0], mask.size[1], tile_size, overlap)

            coverages = []
            effective_sizes = []
            for component in components:
                coverage = tile_component_coverage(mask_bool, component, coords)
                coverages.append(coverage)
                effective_sizes.append(component["area"] * coverage * (tile_resize / tile_size) ** 2)

            if len(effective_sizes) == 0:
                effective_min_component_px = 0.0
                min_coverage = 0.0
            else:
                effective_min_component_px = float(np.min(effective_sizes))
                min_coverage = float(np.min(coverages))

            failed = bool(effective_min_component_px < MIN_VISIBLE_DEFECT_PX or min_coverage < 0.80)

            diag_rows.append({
                "run_id": rule["run_id"],
                "dataset": dataset,
                "category": category,
                "defect_type": row.get("defect_type", "unknown"),
                "strategy": strategy,
                "patchcore_mode": mode,
                "image_path": row["image_path"],
                "mask_path": row["mask_path"],
                "orig_mask_ratio": original["mask_ratio"],
                "orig_min_component_px": original["min_component_px"],
                "after_min_component_px": np.nan,
                "area_survival": np.nan,
                "tile_count": len(coords),
                "effective_min_component_px": effective_min_component_px,
                "failed": failed,
            })

mask_diag_df = pd.DataFrame(diag_rows)

summary_df = (
    mask_diag_df.groupby(["run_id", "dataset", "category", "strategy", "patchcore_mode"])
    .agg(
        masks=("failed", "count"),
        failure_pct=("failed", lambda s: float(s.mean() * 100)),
        median_orig_min_component_px=("orig_min_component_px", "median"),
        median_effective_min_component_px=("effective_min_component_px", "median"),
        mean_tile_count=("tile_count", "mean"),
        mean_area_survival=("area_survival", "mean"),
    )
    .reset_index()
)

print("Rules matrix:")
display(rules_df)
print()
print("Mask diagnostic summary:")
display(summary_df)


Rules matrix:


,run_id,dataset,category,patchcore_mode,strategy,full_image_strategy,tile_strategy,tile_size,tile_overlap,tile_resize,hypothesis
0,steel_resize_512,hss-iad,STEEL,full_image,resize_512,resize_512,,NaN,NaN,NaN,"STEEL full resize, check if tiling was hurting..."
1,steel_letterbox_edge_512,hss-iad,STEEL,full_image,letterbox_edge_512,letterbox_edge_512,,NaN,NaN,NaN,"STEEL full letterbox edge, preserve geometry w..."
2,steel_tile_512,hss-iad,STEEL,tiling,tile_512_overlap_0_50_resize_384,resize_512,tile_512_overlap_0_50_resize_384,512.0,0.5,384.0,STEEL tiling baseline from V4 fast
3,casting1_resize_512,hss-iad,Casting_class1,full_image,resize_512,resize_512,,NaN,NaN,NaN,"Casting class 1 full resize, check normal vers..."
4,casting1_letterbox_edge_512,hss-iad,Casting_class1,full_image,letterbox_edge_512,letterbox_edge_512,,NaN,NaN,NaN,"Casting class 1 letterbox edge, preserve geometry"
5,casting1_tile_256,hss-iad,Casting_class1,tiling,tile_256_overlap_0_50_resize_384,resize_512,tile_256_overlap_0_50_resize_384,256.0,0.5,384.0,"Casting class 1 small tiles, preserve small lo..."
6,cable_resize_512,mvtec,cable,full_image,resize_512,resize_512,,NaN,NaN,NaN,"Cable full resize, baseline without tiling"
7,cable_letterbox_edge_512,mvtec,cable,full_image,letterbox_edge_512,letterbox_edge_512,,NaN,NaN,NaN,"Cable letterbox edge, reduce geometry distortion"
8,cable_tile_512,mvtec,cable,tiling,tile_512_overlap_0_50_resize_384,resize_512,tile_512_overlap_0_50_resize_384,512.0,0.5,384.0,"Cable tiling, check if local patches reduce he..."



Mask diagnostic summary:


,run_id,dataset,category,strategy,patchcore_mode,masks,failure_pct,median_orig_min_component_px,median_effective_min_component_px,mean_tile_count,mean_area_survival
0,cable_letterbox_edge_512,mvtec,cable,letterbox_edge_512,full_image,92,0.000000,27415.0,NaN,NaN,0.999825
1,cable_resize_512,mvtec,cable,resize_512,full_image,92,0.000000,27415.0,NaN,NaN,0.999825
2,cable_tile_512,mvtec,cable,tile_512_overlap_0_50_resize_384,tiling,92,1.086957,27415.0,15365.53125,9.0,NaN
3,casting1_letterbox_edge_512,hss-iad,Casting_class1,letterbox_edge_512,full_image,19,0.000000,97.0,NaN,NaN,1.000442
4,casting1_resize_512,hss-iad,Casting_class1,resize_512,full_image,19,0.000000,97.0,NaN,NaN,1.000442
5,casting1_tile_256,hss-iad,Casting_class1,tile_256_overlap_0_50_resize_384,tiling,19,0.000000,97.0,218.25000,49.0,NaN
6,steel_letterbox_edge_512,hss-iad,STEEL,letterbox_edge_512,full_image,734,100.000000,3525.0,NaN,NaN,0.160435
7,steel_resize_512,hss-iad,STEEL,resize_512,full_image,734,0.272480,3525.0,NaN,NaN,1.001395
8,steel_tile_512,hss-iad,STEEL,tile_512_overlap_0_50_resize_384,tiling,734,5.994550,3525.0,1982.81250,6.0,NaN


## 6. Sauvegarde

In [8]:
##################
#  Save outputs  #
##################

rules_df.to_csv(OUTPUT_RULES_CSV, index=False)
rules_df.to_csv(OUTPUT_RULES_PATCHCORE_CSV, index=False)
mask_diag_df.to_csv(OUTPUT_MASK_DIAG_CSV, index=False)
summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)

print("Saved:", OUTPUT_RULES_CSV)
print("Saved:", OUTPUT_RULES_PATCHCORE_CSV)
print("Saved:", OUTPUT_MASK_DIAG_CSV)
print("Saved:", OUTPUT_SUMMARY_CSV)
print()
print("PatchCore V5 must use:")
print(OUTPUT_RULES_PATCHCORE_CSV)


Saved: /home/nat/Documents/AI-Lab/mar26_bds_anomalies_pieces_indus/notebooks/results/preprocessing_rules_v5_3cat_compare.csv
Saved: /home/nat/Documents/AI-Lab/mar26_bds_anomalies_pieces_indus/notebooks/results/preprocessing_rules_v5_for_patchcore.csv
Saved: /home/nat/Documents/AI-Lab/mar26_bds_anomalies_pieces_indus/notebooks/results/preprocessing_mask_diagnostics_v5_3cat_compare.csv
Saved: /home/nat/Documents/AI-Lab/mar26_bds_anomalies_pieces_indus/notebooks/results/preprocessing_strategy_summary_v5_3cat_compare.csv

PatchCore V5 must use:
/home/nat/Documents/AI-Lab/mar26_bds_anomalies_pieces_indus/notebooks/results/preprocessing_rules_v5_for_patchcore.csv


## 7. Conclusion du notebook 02 V5

Ce notebook ne décide pas à l’avance que le tiling est meilleur. Il prépare une comparaison équitable.

Pour chaque catégorie, PatchCore V5 testera trois stratégies et choisira avec les métriques réelles.

La décision finale se fera avec image AUROC, pixel AUROC, AUPIMO, recall, F1, missed defect, red everywhere et bad localization.